In [ ]:
import yaml
import os
from pathlib import Path
import jax.numpy as jnp
import matplotlib.pyplot as plt
from generative_uncertainty.compute_uncertainty import get_uncertainty_scores 
from generative_uncertainty.plots import plot_uncertainty_threshold_analysis
from generative_uncertainty.config import load_config
from generative_uncertainty.plots import show_scatter_with_threshold
from generative_uncertainty.plots import show_hallucination_analysis
import os
%matplotlib inline
%load_ext autoreload
%autoreload 2

config = "config.yml"
config_data = {}
if os.path.exists(config):
    with open(config, 'r') as f:
        config_data = load_config(config) 
else:
    print(f"Warning: Config file '{config}' not found. Using defaults.")

trained_models_dir = config_data.deep_ensemble.trained_models_dir
real_dataset_path = trained_models_dir.format(seed=0) + "/real_dataset.npy"
real_data = jnp.load(real_dataset_path)
print(f"loaded real data : {real_data.shape}")

samples_cache_dir = config_data.sampling.samples_cache_dir
from shared_utils.colors import okabe_ito as colors

In [ ]:
trained_models_dir = config_data['deep-ensemble']['trained_models_dir']
real_dataset_path = trained_models_dir.format(seed=0) + "/real_dataset.npy"
real_data = jnp.load(real_dataset_path)
print(f"loaded real data : {real_data.shape}")

In [ ]:
samples_cache_dir = config_data['sampling']['samples_cache_dir']
la_sampled_models_dir = Path(config_data['laplace-ensemble']['la_sampled_models_dir'])

In [ ]:
pre_sigma = la_sampled_models_dir / "pre_sigma.npy"
post_sigma = la_sampled_models_dir / "post_sigma.npy"
pre_sigma = jnp.load(pre_sigma)
post_sigma = jnp.load(post_sigma)

In [ ]:
import numpy as np

# Filter out zeros or negative values just in case for log10
valid_pre = pre_sigma[pre_sigma > 0]
valid_post = post_sigma[post_sigma > 0]

plt.hist(np.log10(valid_pre), bins=100, alpha=0.5, label='Log10(Pre Sigma)', density=True)
plt.hist(np.log10(valid_post), bins=100, alpha=0.5, label='Log10(Post Sigma)', density=True)
plt.legend()
plt.title("Log-Scaled Histogram of Pre and Post Sigma Values")
plt.xlabel("Log10(Sigma)")
plt.ylabel("Density")
plt.show()

In [ ]:
# Filter out zeros or negative values just in case for log10
valid_pre = pre_sigma[pre_sigma > 0]
valid_post = post_sigma[post_sigma > 0]

plt.hist(np.log10(valid_pre), bins=100, alpha=0.5, label='Log10(Pre Sigma)', density=True)
plt.hist(np.log10(valid_post), bins=100, alpha=0.5, label='Log10(Post Sigma)', density=True)
plt.legend()
plt.title("Log-Scaled Histogram of Pre and Post Sigma Values")
plt.xlabel("Log10(Sigma)")
plt.ylabel("Density")
plt.show()

In [ ]:
# Filter out zeros or negative values just in case for log10
valid_pre = pre_sigma[pre_sigma > 0]
valid_post = post_sigma[post_sigma > 0]

plt.hist(np.log10(valid_pre), bins=100, alpha=0.5, label='Log10(Pre Sigma)', density=True)
plt.hist(np.log10(valid_post), bins=100, alpha=0.5, label='Log10(Post Sigma)', density=True)
plt.legend()
plt.title("Log-Scaled Histogram of Pre and Post Sigma Values")
plt.xlabel("Log10(Sigma)")
plt.ylabel("Density")
plt.show()

In [ ]:
print(f"Pre Sigma: mean={jnp.mean(pre_sigma):.4f}, std={jnp.std(pre_sigma):.4f}, min={jnp.min(pre_sigma):.4e}, max={jnp.max(pre_sigma):.4e}")
print(f"Post Sigma: mean={jnp.mean(post_sigma):.4f}, std={jnp.std(post_sigma):.4f}, min={jnp.min(post_sigma):.4e}, max={jnp.max(post_sigma):.4e}")
print("shapes:", pre_sigma.shape, post_sigma.shape)

In [ ]:
import glob
# os.listdir("/dtu/blackhole/13/213811/s243425/gaussian_experiment/samples")
models = glob.glob("/dtu/blackhole/13/213811/s243425/gaussian_experiment/models/la_sampled/fisher*")
# samples = glob.glob("/dtu/blackhole/13/213811/s243425/gaussian_experiment/samples/oft_*.npy")
# samples.extend(glob.glob("/dtu/blackhole/13/213811/s243425/gaussian_experiment/samples/lora_*.npy"))
for i, model in enumerate(models):
    print(f"{i}: {model}")

from generative_uncertainty.utils import get_clean_label

In [ ]:
model = np.load(models[2])
plt.hist(model, bins=100)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assuming 'fisher_diagonal' is your 1D numpy array of values
# Sort the values in descending order
sorted_fisher = np.sort(model)[::-1]

plt.figure(figsize=(8, 5))
plt.plot(sorted_fisher, color='#1f77b4', linewidth=2)

# Use a log scale for the Y-axis to reveal the tail
plt.yscale('log') 

plt.ylabel('Fisher Diagonal Value (Log Scale)', fontsize=12)
plt.xlabel('Parameter Index (Sorted by Magnitude)', fontsize=12)
plt.title('Spectrum of the Diagonal Empirical Fisher', fontsize=14)
plt.grid(True, which="both", ls="--", alpha=0.4)

plt.tight_layout()
plt.savefig('fisher_spectrum.png', dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assuming 'fisher_diagonal' is your 1D numpy array
# Filter out exact zeros to avoid log10 errors, or add a tiny epsilon (e.g., 1e-8)
safe_fisher = model[model > 0]

plt.figure(figsize=(8, 5))

# Create logarithmically spaced bins
min_val = np.min(safe_fisher)
max_val = np.max(safe_fisher)
bins = np.logspace(np.log10(min_val), np.log10(max_val), 50)

plt.hist(safe_fisher, bins=bins, color='#1f77b4', edgecolor='black', alpha=0.8)

# Log scale on both axes
plt.xscale('log')
plt.yscale('log')

plt.ylabel('Count (Log Scale)', fontsize=12)
plt.xlabel('Fisher Diagonal Value (Log Scale)', fontsize=12)
plt.title('Distribution of Fisher Diagonal Values', fontsize=14)
plt.grid(True, axis='y', ls="--", alpha=0.4)

plt.tight_layout()
plt.savefig('fisher_histogram_log.png', dpi=300)
plt.show()

In [ ]:
import torch
import numpy as np

# Ensure it's a 1D numpy array for easy analysis
if torch.is_tensor(model):
    fisher_diagonal = model.detach().cpu().numpy().flatten()
else:
    fisher_diagonal = np.array(model).flatten()

# 1. Basic statistical breakdown
total_params = len(fisher_diagonal)
min_val = np.min(fisher_diagonal)
max_val = np.max(fisher_diagonal)
mean_val = np.mean(fisher_diagonal)
median_val = np.median(fisher_diagonal)

# 2. Percentile breakdown (This tells you the shape of the distribution)
# 10%, 25%, 50%, 75%, 90%, 99%, 99.9%
percentiles = np.percentile(fisher_diagonal, [10, 25, 50, 75, 90, 99, 99.9])

# 3. Threshold analysis (How many are 'flat' vs 'sharp'?)
# Let's define "flat" as anything less than 1e-4, and "sharp" as > 100
flat_count = np.sum(fisher_diagonal < 1e-4)
sharp_count = np.sum(fisher_diagonal > 100)

print("--- Fisher Diagonal Statistical Profile ---")
print(f"Total Parameters: {total_params:,}")
print(f"Min Value:        {min_val:.2e}")
print(f"Max Value:        {max_val:.2e}")
print(f"Mean Value:       {mean_val:.2e}")
print(f"Median Value:     {median_val:.2e}")
print("-" * 40)
print(f"10th Percentile:  {percentiles[0]:.2e}")
print(f"50th (Median):    {percentiles[2]:.2e}")
print(f"90th Percentile:  {percentiles[4]:.2e}")
print(f"99th Percentile:  {percentiles[5]:.2e}")
print(f"99.9th Percentile:{percentiles[6]:.2e}")
print("-" * 40)
print(f"Flat directions (< 1e-4): {flat_count:,} ({(flat_count/total_params)*100:.2f}%)")
print(f"Sharp directions (> 100): {sharp_count:,} ({(sharp_count/total_params)*100:.2f}%)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.style.use('seaborn-v0_8-paper')
plt.rcParams.update({
    'font.size': 14,          # General font size
    'axes.titlesize': 16,     # Title size
    'axes.labelsize': 14,     # X and Y axis label size
    'xtick.labelsize': 12,    # X tick label size
    'ytick.labelsize': 12,    # Y tick label size
    'legend.fontsize': 14# Legend text size
})
# Sort the values from largest to smallest
sorted_fisher = np.sort(fisher_diagonal)[::-1]

plt.figure(figsize=(8, 5))
# Use a scatter plot or a line plot for small parameter counts
plt.plot(np.arange(len(sorted_fisher)), sorted_fisher, color='#1f77b4', linewidth=2)
plt.fill_between(np.arange(len(sorted_fisher)), sorted_fisher, 1e-40, color='#1f77b4', alpha=0.2)

# Use a log scale for the Y-axis to reveal the extreme dynamic range
plt.yscale('log') 
plt.ylim(1)

# Draw a red dashed line showing the "danger zone" (< 1e-4)
plt.axhline(y=100, color=colors['black'], linestyle='--', label='Flat Threshold (< 1e2)')
plt.axhline(y=1e-4, color=colors['orange'], linestyle='--', label='Flat Threshold (< 1e-4)')

plt.ylabel('Fisher Diagonal Value (Log Scale)', fontsize=12)
plt.xlabel('Parameter Index (Sorted by Curvature)', fontsize=12)
plt.title('Spectrum of the Diagonal Empirical Fisher', fontsize=14)
plt.legend(loc='lower center', fancybox=True, shadow=True)
plt.grid(True, which="major", ls="-", alpha=0.3)

plt.tight_layout()
plt.savefig('fisher_scree_plot.png', dpi=300)
plt.show()